In [2]:
import os
from pathlib import Path
from typing import Optional

import fastmri
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
from data_utils import *
from datasets import *
from fastmri.data.transforms import tensor_to_complex_np
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, TensorDataset

from model import *
from torch.optim import SGD, Adam, AdamW
from train_utils_meta import *

In [ ]:
path_to_data = '/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/'
n_volumes = 4
n_slices = 2
with_mask = False  # NOTE: During inference phase, set to True.
acceleration = 4
center_frac= 0.15
vol_id0= 0
vol = 0
path_to_data = Path('/itet-stor/mcrespo/bmicdatasets-originals/Originals/fastMRI/brain/multicoil_train/')
dataset = KCoordDataset(path_to_data, n_volumes=2, n_slices=2, with_mask=False, acceleration=4, center_frac=0.15)
center = dataset.metadata[vol]["center"]
left_idx, right_idx, center_vals = center["left_idx"], center["right_idx"], center["vals"]
shape = dataset.metadata[vol]["shape"]
n_slices, n_coils, height, width = shape

# Create tensors of indices for each dimension
kx_ids = torch.cat([torch.arange(left_idx), torch.arange(right_idx, width)])
ky_ids = torch.arange(height)
kz_ids = torch.arange(n_slices)
coil_ids = torch.arange(n_coils)

# Use meshgrid to create expanded grids
kspace_ids = torch.meshgrid(kx_ids, ky_ids, kz_ids, coil_ids, indexing="ij")
kspace_ids = torch.stack(kspace_ids, dim=-1).reshape(-1, len(kspace_ids))

path_to_data = Path(path_to_data)
if path_to_data.is_dir():
    files = sorted(
        [
            file
            for file in path_to_data.iterdir()
            if file.suffix == ".h5" and "AXT1POST_205" in file.name
            # if file.suffix == ".h5" and "AXT2_205" in file.name # T2 sequence
            
        ]
    )[:n_volumes]
with h5py.File(files[0], "r") as hf:
    volume_kspace = to_tensor(preprocess_kspace(hf["kspace"][()]))[
        :n_slices
    ]


In [8]:
coil_dim = 256
vol_dim = 512

#####################################################################
# volume embedding
#####################################################################
embeddings_vol = torch.nn.Embedding(
    len(dataset.metadata), vol_dim
)

## Coil embeddings initialization
########################################################
coil_sizes = []
for i in range(len(dataset.metadata)):
    _, n_coils, _, _ = dataset.metadata[i]["shape"]
    coil_sizes.append(n_coils)
    
total_n_coils = torch.cumsum(torch.tensor(coil_sizes), dim=0)[-1]

# Create the indexes to access the embedding coil table
start_idx = torch.tensor([0] + list(torch.cumsum(torch.tensor(coil_sizes), dim=0)[:-1]))
embeddings_coil = torch.nn.Embedding(total_n_coils.item(), coil_dim)

model_hash_check = '/scratch_net/ken/mcrespo/proj_marina/logs_new/multivol_12_23/2025-01-12_14h20m17s/checkpoints/epoch_0499.pt' # Hash table with coil and volume embeddings
hash_model = torch.load(model_hash_check, map_location=torch.device('cpu'))['model_state_dict']
model = Siren(levels=15, n_min=45, size_hashtable=12, vol_embedding_dim=256, coil_embedding_dim=128, n_features=3, n_layers=8, n_volumes=5)

for layer_name,tensor in model.named_parameters():
    if layer_name in model_hash_check:
        tensor.data.copy_(model_hash_check['model_state_dict'][layer_name])
        
vol_embedding = torch.load(model_hash_check,  map_location=torch.device('cpu'))['embedding_vol_state_dict']['weight'][:2]
coil_embedding = torch.load(model_hash_check,  map_location=torch.device('cpu'))['embedding_coil_state_dict']['weight'][:embeddings_coil.weight.shape[0]]

### Keep the initial tensor for volume and coil
embeddings_vol.weight.data.copy_(vol_embedding)
embeddings_coil.weight.data.copy_(coil_embedding)
embedding_vol_0 = embeddings_vol.weight.detach().clone()
embedding_coil_0 = embeddings_coil.weight.detach().clone()


for param in model.parameters():
    param.requires_grad = False
    
for param in model.embed_fn.parameters():
    param.requires_grad = True

# Only embeddings and Hash encoders are optimized.
optimizer = Adam(
    chain(embeddings_vol.parameters(), embeddings_coil.parameters(), model.embed_fn.parameters(),), lr = 1.e-4)
    

/tmp/ipykernel_27252/1285786786.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  hash_model = torch.load(model_hash_check, map_location=torch.device('cpu'))['model_state

In [11]:
vol_id = 0
limit = 4 # EVALUATED ON 4 POINTS - DISTANCED 1 pixel IN X DIMENSION
target = torch.zeros(limit, 2)
coords = torch.zeros(limit, 4)
inputs = torch.zeros(limit, 5)

for i in range(limit):
    posx = 50+i*10
    posy = 120+i*10
    print(f'Pixel : {posx}, {posy}')
    ## ORDER IS: vol_id, x_coor, ycoor, z_coor, coil_id 
    inputs[i] = torch.tensor([0, posx, posy, 0, i])
    ## ORDER IS: slice, coil_id, ycoor, xcoor 
    target[i]= volume_kspace[0, i, posy, posx, ...]
    coords[i] = inputs[i,:-1]
    
loss_fn = MSELoss(gamma=1)
optimizer.zero_grad(set_to_none=True)
for item in model.parameters():
    item.requires_grad = False
    
vol_ids = inputs[:,0].long()
coil_ids = inputs[:,-1].long()

for i in range(1):
    # Get the index for In the provided code snippet, the variable `t` is not explicitly defined or referenced. It seems to be missing or not relevant to the context provided. If you have more context or specific details about where `t` is used or defined in your code, please provide additional information so that I can assist you better.

    latent_vol = embeddings_vol(vol_ids)
    latent_coil = embeddings_coil(start_idx[vol_id]+coil_ids)

    # optimizer.zero_grad(set_to_none=True)
    outputs = model(coords, latent_vol, latent_coil)
    loss = loss_fn(outputs, target)
    loss.backward()
    optimizer.step()

# GRADIENT CHANGES FOR VOLUME AND COIL EMBEDDINGS OF THE FIRST AND ONLY VOLUME
gradient_vol = embedding_vol_0[0] - embeddings_vol.weight[0]
gradient_coil = embedding_coil_0[:20] - embeddings_coil.weight[:20]
print(f'VOL EMBEDDS dim: {vol_dim}, \nVOL Gradient diff from {i+1} updates: {torch.norm(gradient_vol)}')

norm_coils = []
for tensor in gradient_coil:
    norm_coils.append(torch.norm(tensor.detach()))

print(f'COIL EMBEDDS dim: {coil_dim}, \nCOIL Gradient diff from {i+1} updates: {np.mean(norm_coils)}')

Pixel : 50, 120
Pixel : 60, 130
Pixel : 70, 140
Pixel : 80, 150


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x814 and 430x512)